# 05_gold_checks
Evidence for Phase 3. Run the cells in order AFTER `04_silver_to_gold` has succeeded, and take screenshots.

| Proof | Part |
|---|---|
| Row counts, key uniqueness, foreign keys, reconciliation with Silver | A |
| Performance optimization (OPTIMIZE + Liquid Clustering) | B |
| Query plan / DAG of the most complex transformation | C |
| Sanity check of the business output (top surges, momentum) | D |
| Silver-to-Gold logging and run times | E |

In [0]:
%run ./common

In [0]:
%run ./common_gold

## Part A: data quality of the Gold layer

In [0]:
print("Gold row counts")
for t in GOLD_TABLES:
    print("%-50s %d" % (t, spark.table(t).count()))

In [0]:
# Primary key checks: every number must be 0
KEY_CHECKS = [
    (DIM_DATE, "date_key"),
    (DIM_HOUR, "hour_key"),
    (DIM_EVENT_TYPE, "event_type"),
    (DIM_REPO, "repo_id"),
    (FACT_ACTIVITY_HOURLY, "date_key, hour_key, event_type"),
    (FACT_DAILY_SUMMARY, "date_key"),
    (FACT_REPO_DAILY, "date_key, repo_id"),
    (FACT_REPO_TREND, "date_key, repo_id"),
]
print("Duplicate keys (must be 0)")
for table_name, key_cols in KEY_CHECKS:
    dups = spark.sql(
        "SELECT " + key_cols + ", COUNT(*) AS n FROM " + table_name +
        " GROUP BY " + key_cols + " HAVING COUNT(*) > 1"
    ).count()
    print("%-50s %-32s %d" % (table_name, key_cols, dups))

In [0]:
# Foreign key checks: rows in a fact whose key is missing from the dimension. Every number must be 0.
FK_CHECKS = [
    (FACT_ACTIVITY_HOURLY, "date_key", DIM_DATE, "date_key"),
    (FACT_ACTIVITY_HOURLY, "hour_key", DIM_HOUR, "hour_key"),
    (FACT_ACTIVITY_HOURLY, "event_type", DIM_EVENT_TYPE, "event_type"),
    (FACT_DAILY_SUMMARY, "date_key", DIM_DATE, "date_key"),
    (FACT_REPO_DAILY, "date_key", DIM_DATE, "date_key"),
    (FACT_REPO_DAILY, "repo_id", DIM_REPO, "repo_id"),
    (FACT_REPO_TREND, "date_key", DIM_DATE, "date_key"),
    (FACT_REPO_TREND, "repo_id", DIM_REPO, "repo_id"),
]
print("Orphan foreign keys (must be 0)")
for fact, fact_col, dim, dim_col in FK_CHECKS:
    orphans = spark.sql(
        "SELECT f." + fact_col + " FROM " + fact + " f LEFT ANTI JOIN " + dim + " d ON f." + fact_col + " = d." + dim_col
    ).count()
    print("%-45s %-12s -> %-35s %d" % (fact, fact_col, dim, orphans))

In [0]:
# Reconciliation: Gold must contain exactly the events that are in Silver
silver_total = spark.sql("SELECT COUNT(*) AS n FROM " + SILVER_EVENT + " WHERE event_date IS NOT NULL AND repo_id IS NOT NULL").collect()[0]["n"]
hourly_total = spark.sql("SELECT SUM(events) AS n FROM " + FACT_ACTIVITY_HOURLY).collect()[0]["n"]
daily_total = spark.sql("SELECT SUM(events) AS n FROM " + FACT_DAILY_SUMMARY).collect()[0]["n"]
print("silver_event rows           :", silver_total)
print("sum of fact_activity_hourly :", hourly_total)
print("sum of fact_daily_summary   :", daily_total)
print("MATCH" if silver_total == hourly_total and silver_total == daily_total else "MISMATCH")

In [0]:
# Share of activity covered by tracked repositories (the size decision for dim_repo)
display(spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM """ + DIM_REPO + """)                                   AS tracked_repos,
        (SELECT COUNT(DISTINCT repo_id) FROM """ + SILVER_EVENT + """)                AS all_repos,
        (SELECT SUM(total_events) FROM """ + FACT_REPO_DAILY + """)                   AS tracked_repo_events,
        (SELECT COUNT(*) FROM """ + SILVER_EVENT + """)                               AS all_events
"""))

In [0]:
# Data completeness per day (explains gaps in the charts and which days the surge rules use)
display(spark.sql(
    "SELECT full_date, day_name, hours_loaded, is_complete_day FROM " + DIM_DATE +
    " WHERE hours_loaded > 0 ORDER BY full_date"
))

## Part B: performance optimization
The big fact tables and `dim_repo` use Liquid Clustering (set in `00b_setup_gold`).
`OPTIMIZE` compacts small files and clusters the data by the clustering keys, so Power BI queries that filter on a date
or a repository read fewer files (data skipping).

In [0]:
CLUSTERED_TABLES = [FACT_ACTIVITY_HOURLY, FACT_REPO_DAILY, FACT_REPO_TREND, DIM_REPO]

print("BEFORE optimize")
for t in CLUSTERED_TABLES:
    d = spark.sql("DESCRIBE DETAIL " + t).collect()[0].asDict()
    print("%-50s files=%s  bytes=%s  clustering=%s" % (t, d.get("numFiles"), d.get("sizeInBytes"), d.get("clusteringColumns")))

In [0]:
for t in CLUSTERED_TABLES:
    print("OPTIMIZE", t)
    display(spark.sql("OPTIMIZE " + t))

In [0]:
print("AFTER optimize")
for t in CLUSTERED_TABLES:
    d = spark.sql("DESCRIBE DETAIL " + t).collect()[0].asDict()
    print("%-50s files=%s  bytes=%s  clustering=%s" % (t, d.get("numFiles"), d.get("sizeInBytes"), d.get("clusteringColumns")))

## Part C: query plan of the most complex transformation
This is the core of `fact_repo_daily` + `fact_repo_trend` written as one SQL query:
filter Silver by date, keep tracked repos (semi join), aggregate per repo and day, then a window per repository.

1. Run the next cell. Under the result, click **See performance** (or the query link) to open the **query profile**.
   Screenshot the graph. Serverless compute does not expose the classic Spark UI, the query profile is the
   serverless view of the same DAG.
2. Run the EXPLAIN cell and screenshot the physical plan. Look for: the scan with **PushedFilters / data filters**
   on `event_date` (predicate pushdown), the **Exchange / Shuffle** before the aggregation and before the window
   (data is redistributed by `repo_id`), the **join** with `dim_repo` (broadcast if small), and the **Window** operator.
   On serverless the operators are Photon versions (names start with Photon).

In [0]:
from datetime import timedelta

last_date = spark.sql("SELECT MAX(event_date) AS d FROM " + SILVER_EVENT).collect()[0]["d"]
filter_from = last_date - timedelta(days=13)
print("Query window:", filter_from, "to", last_date)

DAG_QUERY = """
WITH daily AS (
    SELECT e.repo_id, e.event_date, COUNT(*) AS events, COUNT(DISTINCT e.actor_id) AS contributors
    FROM """ + SILVER_EVENT + """ e
    LEFT SEMI JOIN """ + DIM_REPO + """ r ON e.repo_id = r.repo_id
    WHERE e.event_date >= DATE'""" + str(filter_from) + """'
    GROUP BY e.repo_id, e.event_date
)
SELECT repo_id, event_date, events, contributors,
       AVG(events) OVER (PARTITION BY repo_id ORDER BY event_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS previous_avg
FROM daily
ORDER BY events DESC
LIMIT 20
"""
display(spark.sql(DAG_QUERY))

In [0]:
plan_rows = spark.sql("EXPLAIN FORMATTED " + DAG_QUERY).collect()
print(plan_rows[0][0])

## Part D: business output sanity check

In [0]:
print("Surges by type")
display(spark.sql("""
    SELECT surge_type, COUNT(*) AS repo_days, COUNT(DISTINCT repo_id) AS repos
    FROM """ + FACT_REPO_TREND + """
    WHERE is_surge
    GROUP BY surge_type
"""))

In [0]:
print("Top 15 spikes: active repos far above their own 7 day baseline")
display(spark.sql("""
    SELECT d.full_date, r.repo_name, t.events, ROUND(t.baseline_avg, 1) AS baseline_avg,
           ROUND(t.baseline_std, 1) AS baseline_std, ROUND(t.z_score, 1) AS z_score
    FROM """ + FACT_REPO_TREND + """ t
    JOIN """ + DIM_REPO + """ r ON t.repo_id = r.repo_id
    JOIN """ + DIM_DATE + """ d ON t.date_key = d.date_key
    WHERE t.surge_type = 'Spike'
    ORDER BY t.z_score DESC
    LIMIT 15
"""))

In [0]:
print("Top 15 breakouts: quiet repos that suddenly became busy")
display(spark.sql("""
    SELECT d.full_date, r.repo_name, t.events, ROUND(t.baseline_avg, 1) AS baseline_avg,
           f.distinct_actors AS contributors_that_day, f.watch_events AS stars_that_day
    FROM """ + FACT_REPO_TREND + """ t
    JOIN """ + FACT_REPO_DAILY + """ f ON t.date_key = f.date_key AND t.repo_id = f.repo_id
    JOIN """ + DIM_REPO + """ r ON t.repo_id = r.repo_id
    JOIN """ + DIM_DATE + """ d ON t.date_key = d.date_key
    WHERE t.surge_type = 'Breakout'
    ORDER BY t.events DESC
    LIMIT 15
"""))

In [0]:
# Momentum: last 7 complete days against the 7 days before, per tracked repo.
# Computed from fact_repo_daily so repos that were quiet on the last day are still included.
print("Fastest growing tracked repos (at least 50 events in the previous 7 days)")
display(spark.sql("""
    WITH anchor AS (
        SELECT MAX(full_date) AS a FROM """ + DIM_DATE + """ WHERE is_complete_day
    ),
    per_repo AS (
        SELECT f.repo_id,
               SUM(CASE WHEN d.full_date > date_sub(an.a, 7) AND d.full_date <= an.a
                        THEN f.total_events ELSE 0 END) AS last_7d,
               SUM(CASE WHEN d.full_date > date_sub(an.a, 14) AND d.full_date <= date_sub(an.a, 7)
                        THEN f.total_events ELSE 0 END) AS prev_7d
        FROM """ + FACT_REPO_DAILY + """ f
        JOIN """ + DIM_DATE + """ d ON f.date_key = d.date_key
        CROSS JOIN anchor an
        GROUP BY f.repo_id
    )
    SELECT r.repo_name, p.last_7d, p.prev_7d,
           ROUND((p.last_7d - p.prev_7d) * 100.0 / p.prev_7d, 1) AS growth_pct
    FROM per_repo p
    JOIN """ + DIM_REPO + """ r ON p.repo_id = r.repo_id
    WHERE p.prev_7d >= 50
    ORDER BY growth_pct DESC
    LIMIT 15
"""))

## Part E: Silver-to-Gold logging

In [0]:
display(spark.sql("""
    SELECT layer, target_table, parameter_processed, load_type, status, rows_inserted, start_time, end_time, error_message
    FROM """ + EXEC_LOG + """
    WHERE layer = 'Silver-to-Gold'
    ORDER BY start_time DESC
    LIMIT 20
"""))

In [0]:
display(spark.sql("""
    SELECT target_table, date_from, date_to, source_rows_read, rows_written, ROUND(duration_seconds, 1) AS seconds, start_time
    FROM """ + GOLD_METRICS + """
    ORDER BY start_time DESC
    LIMIT 20
"""))

In [0]:
# Run time per layer per day (input for the FinOps section of the report).
# First one row per run and layer (first start to last end), then summed per day.
display(spark.sql("""
    WITH per_run AS (
        SELECT run_id, layer, MIN(start_time) AS run_start, MAX(end_time) AS run_end
        FROM """ + EXEC_LOG + """
        WHERE status = 'Success'
        GROUP BY run_id, layer
    )
    SELECT to_date(run_start) AS run_date, layer,
           COUNT(*) AS runs,
           ROUND(SUM(unix_timestamp(run_end) - unix_timestamp(run_start)) / 60.0, 1) AS minutes
    FROM per_run
    GROUP BY to_date(run_start), layer
    ORDER BY run_date DESC, layer
"""))